In [0]:
# dim ics gold

silver_workforce = (spark.table("silver_workforce_fte"))
silver_workforce.show(10)



In [0]:

from pyspark.sql.functions import col, lower,trim, row_number
from pyspark.sql.window import Window

window_spec = Window.orderBy("ics_code")

dim_ics_gold = (silver_workforce
.select(col("ics_code"), col("ics_name"))
.filter(
        (col("ics_code").isNotNull()) &
        (lower(trim(col("ics_code"))) != "all ics areas")
    )
.distinct()
.withColumn( "ics_key", row_number().over(window_spec))
.select("ics_key", "ics_code", "ics_name"))
dim_ics_gold.show()


In [0]:
# write to gold blob

(dim_ics_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "path",
        "abfss://gold@jdnhsbronze.dfs.core.windows.net/dim_ics/"
    ) \
    .saveAsTable(
        "ics_dimension"
    ))